# Bronze: Local Authority Revenue Expenditure
**Source:** MHCLG LA Revenue Account Outturn
**Purpose:** Load raw CSV into Bronze Delta table
**Date loaded:** [7th April 2026]
**Rows expected:** approximately 300 (one per local authority)

In [2]:
df = spark.read.csv('Files/la_revenue_expenditure.csv', header=True,
inferSchema=True)
print(f'Rows: {df.count()}')
print(f'Columns: {len(df.columns)}')
df.printSchema()

StatementMeta(, d19ff4b9-8eac-41dd-85db-0e517e6af897, 5, Finished, Available, Finished, False)

Rows: 3504
Columns: 2323
root
 |-- year_ending: integer (nullable = true)
 |-- ONS_code: string (nullable = true)
 |-- LA_LGF_code: string (nullable = true)
 |-- LA_name: string (nullable = true)
 |-- status: string (nullable = true)
 |-- LA_class: string (nullable = true)
 |-- LA_subclass: string (nullable = true)
 |-- RG_grantindsg_tot_grant: integer (nullable = true)
 |-- RG_grantinppg_tot_grant: integer (nullable = true)
 |-- RG_grantinusm_tot_grant: integer (nullable = true)
 |-- RG_grantinph_tot_grant: integer (nullable = true)
 |-- RG_grantinscs_tot_grant: integer (nullable = true)
 |-- RG_grantinrsd_tot_grant: integer (nullable = true)
 |-- RG_grantinibcf_tot_grant: integer (nullable = true)
 |-- RG_grantinlts_tot_grant: integer (nullable = true)
 |-- RG_grantinilf_tot_grant: integer (nullable = true)
 |-- RG_grantinmkt_tot_grant: integer (nullable = true)
 |-- RG_grantinsrvs_tot_grant: integer (nullable = true)
 |-- RG_grantinfhs_tot_grant: integer (nullable = true)
 |-- RG_gr

In [3]:
df.show(5, truncate=False)

StatementMeta(, d19ff4b9-8eac-41dd-85db-0e517e6af897, 6, Finished, Available, Finished, False)

+-----------+---------+-----------+---------------------+---------+-----------------+-----------------+-----------------------+-----------------------+-----------------------+----------------------+-----------------------+-----------------------+------------------------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------+-----------------------+-----------------------+------------------------+-----------------------+-------------------------+-----------------------+-----------------------+--------------------------+--------------------------+---------------------------+---------------------------+---------------------------+------------------------+------------------------+---------------------+------------------+---------------------+-----------------------------+---------------------------+---------------------------+----------------------------+-----------------------------+-----------------------------+-----------

In [4]:
key_cols = ['year_ending', 'ONS_code', 'LA_name', 'LA_class', 
            'RS_edu_net_exp', 'RS_asc_net_exp', 'RS_hous_net_exp',
            'RS_totsx_net_exp']

df.select(key_cols).show(5, truncate=False)

StatementMeta(, d19ff4b9-8eac-41dd-85db-0e517e6af897, 7, Finished, Available, Finished, False)

+-----------+---------+---------------------+-----------------+--------------+--------------+---------------+----------------+
|year_ending|ONS_code |LA_name              |LA_class         |RS_edu_net_exp|RS_asc_net_exp|RS_hous_net_exp|RS_totsx_net_exp|
+-----------+---------+---------------------+-----------------+--------------+--------------+---------------+----------------+
|201803     |E06      |Unitary Authorities  |Unitary Authority|6629859       |3613703       |356427         |16330288        |
|201803     |E06000001|Hartlepool UA        |Unitary Authority|58214         |29299         |2940           |153606          |
|201803     |E06000002|Middlesbrough UA     |Unitary Authority|79059         |44121         |5731           |215961          |
|201803     |E06000003|Redcar & Cleveland UA|Unitary Authority|74139         |41160         |2505           |198998          |
|201803     |E06000004|Stockton-on-Tees UA  |Unitary Authority|100788        |51139         |3618           |25

In [5]:
from pyspark.sql.functions import col, count, when

key_cols = ['year_ending', 'ONS_code', 'LA_name', 'LA_class',
            'RS_edu_net_exp', 'RS_asc_net_exp', 'RS_hous_net_exp',
            'RS_csc_net_exp', 'RS_trans_net_exp', 'RS_totsx_net_exp']

null_counts = df.select([count(when(col(c).isNull(), c)).alias(c) for c in key_cols])
null_counts.show()

StatementMeta(, d19ff4b9-8eac-41dd-85db-0e517e6af897, 8, Finished, Available, Finished, False)

+-----------+--------+-------+--------+--------------+--------------+---------------+--------------+----------------+----------------+
|year_ending|ONS_code|LA_name|LA_class|RS_edu_net_exp|RS_asc_net_exp|RS_hous_net_exp|RS_csc_net_exp|RS_trans_net_exp|RS_totsx_net_exp|
+-----------+--------+-------+--------+--------------+--------------+---------------+--------------+----------------+----------------+
|          0|       0|      0|       0|            23|            23|             23|            23|              23|              23|
+-----------+--------+-------+--------+--------------+--------------+---------------+--------------+----------------+----------------+



In [6]:
df.write.format('delta').mode('overwrite').saveAsTable('bronze_la_expenditure')

StatementMeta(, d19ff4b9-8eac-41dd-85db-0e517e6af897, 9, Finished, Available, Finished, False)